In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, time, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [3]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = 'USR_GSIT_GCHIARELLA'
pw = 'eBLfq$GQ6ZJ83P'
hostname='172.20.0.185'
service_name="dbsas"
port = 1521

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [4]:

query="""SELECT 
    CASE
        WHEN I.COD_ETIPO_OPERACIO = '100200' THEN 'ALTA'
        WHEN I.COD_ETIPO_OPERACIO = '200200' THEN 'BAJA'
    END AS TRANSACCION,
    TO_DATE(I.FEC_REGISTRO_SISTEMA, 'YYYYMMDD') AS FECHA,
    COUNT(I.COD_ETIPO_OPERACIO) AS CANTIDAD
FROM 
    USRCSA.CSATINSCRI I
WHERE 
    I.COD_ETIPO_OPERACIO IN ('100200', '200200')
GROUP BY 
    CASE
        WHEN I.COD_ETIPO_OPERACIO = '100200' THEN 'ALTA'
        WHEN I.COD_ETIPO_OPERACIO = '200200' THEN 'BAJA'
    END,
    TO_DATE(I.FEC_REGISTRO_SISTEMA, 'YYYYMMDD')

UNION ALL

SELECT 
    CASE
        WHEN V.ACRCMOT = '74' AND V.ACRITDA IN ('25', '26') THEN 'ACRED_COMP'
    END AS TRANSACCION,
    TO_DATE(TO_CHAR(V.ACRFACTC, 'YYYYMMDD'), 'YYYYMMDD') AS FECHA,
    COUNT(V.ACRFACTC) AS CANTIDAD
FROM 
    SIA.SCCVDACR@LNK_SIA_SAS.ESSALUD V
WHERE 
    V.ACRCMOT = '74'   AND V.ACRITDA IN ('25', '26')
GROUP BY 
    CASE
        WHEN V.ACRCMOT = '74' AND V.ACRITDA IN ('25', '26') THEN 'ACRED_COMP'
    END,
    TO_DATE(TO_CHAR(V.ACRFACTC, 'YYYYMMDD'), 'YYYYMMDD')
    
UNION ALL


SELECT 
    CASE
        WHEN COD_ETIPO_SUBSIDIO = 'LT' THEN 'LACTANCIA'
        WHEN COD_ETIPO_SUBSIDIO = 'MA' THEN 'MATERNIDAD'
        WHEN COD_ETIPO_SUBSIDIO = 'IN' THEN 'INCAPACIDAD'
        WHEN COD_ETIPO_SUBSIDIO = 'CC' THEN 'CANJE_CMP'
        WHEN COD_ETIPO_SUBSIDIO = 'SP' THEN 'RECUPER_EMPLEO'
        WHEN COD_ETIPO_SUBSIDIO = 'SL' THEN 'SUSP_PERFEC_LAB'
        WHEN COD_ETIPO_SUBSIDIO = 'CD' THEN 'INCAPC_COVID' 
    END AS TRANSACCION,
    TO_DATE(TO_CHAR(FEC_USUARIO_CREA, 'YYYYMMDD'), 'YYYYMMDD') AS FECHA,
    COUNT(FEC_USUARIO_CREA) AS CANTIDAD
FROM 
    ospe_virtual.ssutsolici_ov su
WHERE 
    COD_ETIPO_SUBSIDIO IN ('LT', 'MA', 'IN', 'CC', 'SP', 'SL', 'CD')
GROUP BY 
    CASE
        WHEN COD_ETIPO_SUBSIDIO = 'LT' THEN 'LACTANCIA'
        WHEN COD_ETIPO_SUBSIDIO = 'MA' THEN 'MATERNIDAD'
        WHEN COD_ETIPO_SUBSIDIO = 'IN' THEN 'INCAPACIDAD'
        WHEN COD_ETIPO_SUBSIDIO = 'CC' THEN 'CANJE_CMP'
        WHEN COD_ETIPO_SUBSIDIO = 'SP' THEN 'RECUPER_EMPLEO'
        WHEN COD_ETIPO_SUBSIDIO = 'SL' THEN 'SUSP_PERFEC_LAB'
        WHEN COD_ETIPO_SUBSIDIO = 'CD' THEN 'INCAPC_COVID' 
    END,
    TO_DATE(TO_CHAR(FEC_USUARIO_CREA, 'YYYYMMDD'), 'YYYYMMDD')

UNION ALL

SELECT 
    CASE
        WHEN SH.TXT_IPUSUARIO_CREA='10.2.5.51' THEN 'REACTIVACION'
    END AS TRANSACCION,
    TO_DATE(TO_CHAR(SH.FEC_USUARIO_CREA, 'YYYYMMDD'), 'YYYYMMDD') AS FECHA,
    COUNT(SH.FEC_USUARIO_CREA) AS CANTIDAD
FROM 
    USRCSA.SSUHREAC SH
WHERE 
    SH.TXT_IPUSUARIO_CREA='10.2.5.51'
GROUP BY 
    CASE
        WHEN SH.TXT_IPUSUARIO_CREA='10.2.5.51' THEN 'REACTIVACION'
    END,
    TO_DATE(TO_CHAR(SH.FEC_USUARIO_CREA, 'YYYYMMDD'), 'YYYYMMDD')"""

base1 = pd.read_sql_query(query, con=connection0)

connection0.close()
engine0.dispose()

base1.to_sql(name=f'conteo_transacciones_diarias', con=engine2, if_exists='replace', index=False)

235

In [ ]:
#4 12 10